# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

#### Business Objective:
Help a used car dealership make informed decisison by identifying which attributes of a used car most stronlgly influence its sale price.

#### Business Success Criteria:
The analysis is sconsidered successful if it produces clear, actionable recommendationss that the dealership can use to decide which types of cars to stock.

#### Resources:
A dataset of 426K cars sourced from Kaggle (a subset of 3-million record original dataset).

#### Requirements:
* No additional data collection is planned.
* Results must be delivered as a Jupyter notebook with accompanying README summary.
* Findings must be in plain-langauage recommendations for a non-technical audience.

#### Data Mining Goal:
* Build a supervised regression model that predicts used car prices based on vehicle attributes. The model must quantify the relative importance and direction of the different features's effect on the car price.

#### Data Mining Success Criteria:
* Low RMSE on held-out data via cross-validation

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

### Collect Initial Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('data/vehicles.csv')

# Confirm successful load by previewing the first few rows
print("Dataset loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

### Describe Data

In [ ]:
df.dropna().head()

In [ ]:
df.info()

#### Data Description Report:
* There are 2 identifiers: `id` and `VIN` (Vehicle Identification Number) which should be a unique, 17-character alphanumeric fingerprint assigned to every motor vehicle.
* The Target is the `price` column
* 2 Numeric Features: 
1. `year` and 
2. `odometer`
* 14 Categorical Features: 
1. `region`
2. `manufacturer`
3. `model`
4. `condition`
5. `cylinders`
6. `fuel`
7. `title_status`
8. `transmission`
9. `drive`
10. `size`
11. `type`
13. `paint_color`
14. `state`



In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})\
  .query('`Missing Count` > 0')\
  .sort_values('Missing %', ascending=False)

In [ ]:
df['price'].describe()
print(f"Zero prices: {(df['price'] == 0).sum():,}")
print(f"Prices > $100,000: {(df['price'] > 100_000).sum():,}")
print(f"Max price: ${df['price'].max():,.0f}")

#### Data Quality Report
* 72% of `size` data is missing. I will need to drop it as the available remaining data won't be useful.
* Although `VIN` number encodes specific vehicle information but 38% of them are already missing and decoding it requires manufacturer-specific lookup table that is not available to use. I will drop this column and rely on the `id` as the identifier.
* 32,895 listings have a price of $0. These are important missing data so the records need to be removed.
* 174,104 (41%) of the records have missing `condition`, but we should keep it although that means that we would have to drop the missing records which would leave us with 142,117 records (33.3% of the original dataset) but that should remain sufficient for robust modeling.
* Odometer max was 10,000,000 which is clearly an error

In [ ]:
df.describe()

In [ ]:
# Check if id is truly unique
print(f"Total rows: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
print(f"Duplicates: {df['id'].duplicated().sum()}")

In [ ]:
# Check if there are records with zero as the Price
df['price'].describe()
print(f"Zero prices: {(df['price'] == 0).sum():,}")
print(f"Prices > $100,000: {(df['price'] > 100_000).sum():,}")
print(f"Max price: ${df['price'].max():,.0f}")

In [ ]:
# Check for the best approach to deal with the missing data by calculating how many rows would remains 
original = len(df)
print(f"Original:                    {original:,}  (100%)")

cols_to_drop = ['size', 'VIN', 'cylinders']
df_temp = df.drop(columns=cols_to_drop)
print(f"After dropping columns:      {len(df_temp):,}  ({len(df_temp)/original*100:.1f}%)")


missing_temp = df_temp.isnull().sum()
missing__temp_pct = (df_temp.isnull().sum() / len(df_temp) * 100).round(2)
pd.DataFrame({'Missing Count': missing_temp, 'Missing %': missing__temp_pct})\
  .query('`Missing Count` > 0')\
  .sort_values('Missing %', ascending=False)

In [ ]:
df_temp = df_temp.dropna()
print(f"After dropping NaN rows:     {len(df_temp):,}  ({len(df_temp)/original*100:.1f}%)")

#### Data Exploration Report
* Price Distribution is Heavily Right-Skewed. The distribution of used car prices is not normal. It has extreme outliers. Maximum is $3.7 billion. The median price is $12,995 while the mean is $54,255 due to these outliers.
* Interestingly, vehicles listed as 'good' show a higher median price than those listed as 'excellent' or 'like new'. This suggests that condition alone does not fully explain price. Vehicle type and other features likely play a role. A multivariate regression model will probably be better than relying on any single feature.
* Price tends to decrease as mileage and vehicle age increase, which is expected.
* `title_status` has a clear impact on the price and will be an important feature to retain

In [ ]:
df['price'].describe()

In [ ]:
# How many cars are priced at $0 or very low?
print(f"Price = $0:          {(df['price'] == 0).sum():,}")
print(f"Price < $500:        {(df['price'] < 500).sum():,}")

# How many cars are priced suspiciously high?
print(f"Price > $100,000:    {(df['price'] > 100000).sum():,}")
print(f"Price > $200,000:    {(df['price'] > 200000).sum():,}")
print(f"Price > $1,000,000:  {(df['price'] > 1000000).sum():,}")

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

# Filter out price outliers for cleaner visualizations
# Keeping prices between $500 and $100,000
df_viz = df[(df['price'] > 500) & (df['price'] < 100000)].copy()
df_viz['age'] = 2024 - df_viz['year']

In [ ]:
# Price distribution — reveals skewness and outliers
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df_viz['price'], bins=60)
ax.set_title('Distribution of Used Car Prices', fontsize=13, fontweight='bold')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price Vs Condition
df['condition'].unique()

In [ ]:
condition_order = ['salvage', 'fair', 'good', 'excellent', 'like new', 'new']

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_viz, x='condition', y='price', order=condition_order, ax=ax)

# Add median labels on top of each box
medians = df_viz.groupby('condition')['price'].median()
for i, condition in enumerate(condition_order):
    median_val = medians[condition]
    ax.text(i, median_val, f'${median_val:,.0f}', 
            ha='center', va='bottom', fontweight='bold', fontsize=9, color='black')

ax.set_title('Price by Vehicle Condition', fontsize=13, fontweight='bold')
ax.set_xlabel('Condition')
ax.set_ylabel('Price ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price Vs Title Status
df['title_status'].unique()

In [ ]:

title_status_order = ['missing', 'parts only', 'salvage', 'rebuilt', 'lien' , 'clean']

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_viz, x='title_status', y='price', order=title_status_order, ax=ax)
# Add median labels on top of each box
medians = df_viz.groupby('title_status')['price'].median()
for i, title_status in enumerate(title_status_order):
    median_val = medians[title_status]
    ax.text(i, median_val, f'${median_val:,.0f}', 
            ha='center', va='bottom', fontweight='bold', fontsize=9, color='black')
ax.set_title('Price by Title Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Title Status')
ax.set_ylabel('Price ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price Vs Odometer
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df_viz['odometer'], df_viz['price'], alpha=0.1, s=5)
ax.set_title('Price vs Odometer Reading', fontsize=13, fontweight='bold')
ax.set_xlabel('Odometer (miles)')
ax.set_ylabel('Price ($)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
#Price vs Vehicle Age
df_viz['age'] = 2024 - df_viz['year']

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df_viz['age'], df_viz['price'], alpha=0.1, s=5)
ax.set_title('Price vs Vehicle Age (Years)', fontsize=13, fontweight='bold')
ax.set_xlabel('Vehicle Age (years)')
ax.set_ylabel('Price ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price Vs Fuel Type
fuel_median = df_viz.groupby('fuel')['price'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(fuel_median.index, fuel_median.values)
ax.set_title('Median Price by Fuel Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Fuel Type')
ax.set_ylabel('Median Price ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price Vs Vehicle Type
type_median = df_viz.groupby('type')['price'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(type_median.index, type_median.values)
ax.set_title('Median Price by Vehicle Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Median Price ($)')
ax.set_ylabel('Vehicle Type')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

#### Select Data + Rationale for inclusion/exclusion
The following steps were taken to prepare the dataset for modeling.

---

1. Drop Irrelevant Columns 
- `id` : Unique identifier (not a predictive feature)
- `VIN` : Identifier with 37.7% missing values
- `size` : 71.8% missing (not recoverable)
- `cylinders` : 41.6% missing
- `region` : 404 unique values (too granular). `state` captures geography more cleanly

---

2. Drop Rows with Missing Values
After removing the most problematic columns, remaining rows with any missing values are dropped.
This retains **33.3% of the original dataset (142,117 records)**, which should be sufficient for robust modeling.

---

3. Remove Outliers by applying filters
- `price` (Filter applied: $2,000 – $50,000): Below $500 are likely errors; above $50,000 is only 3% of listings.
- `odometer` (Filter applied: ≤ 200,000 miles): Values above this are clearly erroneous.
- `year` (Filter applied: ≥ 1990): Older vehicles follow different pricing rules (antique/collector market).

---

4. Feature Engineering
- **`age`** is created as `2026 - year` to directly represent vehicle depreciation.
- **`year`** is dropped after `age` is created since they capture the same information.

---

5. Encoding Categorical Variables
Two encoding strategies are used depending on whether a natural order exists between categories:

**Ordinal Encoding**: used when categories have a meaningful order:
- `condition`: `salvage(0) → fair(1) → good(2) → excellent(3) → like new(4) → new(5)`
- `title_status`: `parts only(0) → missing(1) → salvage(2) → rebuilt(3) → lien(4) → clean(5)`

**One-Hot Encoding**: used when no natural order exists between categories:
- Applied to: `manufacturer`, `fuel`, `transmission`, `drive`, `type`, `paint_color`, `state`
- `model` is dropped entirely due to its 29,649 unique values which would add noise rather than signal

---

6. Log-Transform the Target Variable (`price`)
- Used car prices are heavily right-skewed
- Linear regression assumes the target is approximately normally distributed
- Applying `log(price)` compresses the scale, reduces the impact of extreme values, and produces a near-normal distribution

---

7. Scale Numeric Features
- `age` and `odometer` are scaled using **StandardScaler** (mean = 0, standard deviation = 1)

In [ ]:
# These columns add no predictive value:
# - 'id': unique identifier, not a feature
# - 'VIN': identifier, 37.7% missing anyway
# - 'size': 71.8% missing, not recoverable
# - 'cylinders': 41.6% missing
# - 'region': superseded by 'state' which is cleaner

cols_to_drop = ['id', 'VIN', 'size', 'cylinders', 'region']
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
# Remove Rows with Missing Values
# drop rows with any missing values
print(f"Before: {len(df):,}")
original_length = len(df)
df.dropna(inplace=True)
print(f"After:  {len(df):,}  ({len(df)/original_length*100:.1f}% of original)")

In [ ]:
# Remove Price Outliers
# $0 prices are data entry errors
# Prices above $50,000 are small % of listings and will distort modeling
print(f"Before: {len(df):,}")
df = df[(df['price'] >= 2000) & (df['price'] <= 50000)]
print(f"After:  {len(df):,}")

In [ ]:
# Remove Odometer Outliers
# Odometer max was 10,000,000 which is clearly an error
# Keeping values up to 200,000 miles covers even extreme high-mileage cases
print(f"Before: {len(df):,}")
df = df[df['odometer'] <= 200000]
print(f"After:  {len(df):,}")

In [ ]:
# Filter classic/antique cars
# Year min was 1900. Since classic/antique cars follow different pricing rules, we will filter them out
# Keeping 1990+ focuses on the mainstream used car market
print(f"Before: {len(df):,}")
df = df[df['year'] >= 1990]
print(f"After:  {len(df):,}")

In [ ]:
# 'Age' instead of 'Year'
# 'year' as a raw number is less intuitive for a model than age
df['age'] = 2026 - df['year']

# We can now drop 'year' since 'age' captures the same information
df.drop(columns=['year'], inplace=True)

In [ ]:
#Encode Categorical Variables
# 'condition' has a clear order from worst to best
condition_order = ['salvage', 'fair', 'good', 'excellent', 'like new', 'new']

# Map each level to a number preserving the order
condition_map = {c: i for i, c in enumerate(condition_order)}
df['condition_encoded'] = df['condition'].map(condition_map)
df.drop(columns=['condition'], inplace=True)

# title_status also has an implied order by desirability
title_order = ['parts only', 'missing', 'salvage', 'rebuilt', 'lien', 'clean']
title_map = {t: i for i, t in enumerate(title_order)}
df['title_status_encoded'] = df['title_status'].map(title_map)
df.drop(columns=['title_status'], inplace=True)

In [ ]:
# These columns have no meaningful order between categories
nominal_cols = ['manufacturer', 'fuel', 'transmission', 'drive', 'type', 'paint_color', 'state']

# drop_first=True drops one category per column to avoid multicollinearity
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

In [ ]:
# dropping 'model' since it has 29,649 unique values (one-hot encoding would create thousands of columns)
df.drop(columns=['model'], inplace=True)

In [ ]:
# Log-Transform the 'price' Variable as it is heavily right-skewed
df['log_price'] = np.log(df['price'])
df.drop(columns=['price'], inplace=True)

In [ ]:
# Scale numeric values
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'odometer']

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [ ]:
# Final Dataset
print(f"Final shape: {df.shape}")
print(f"\nAny nulls remaining: {df.isnull().sum().sum()}")
print(f"\nTarget variable (log_price) distribution:")
print(df['log_price'].describe())


df.head()

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

#### Modeling Technique
- Linear Regression: Basesline and starting point
- Ridge Regression (L2): Improves on baseline and keeps all features
- Lasso Regression (L1): Improves on baseline and performs feature selection

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error

In [ ]:
# Divide the data into training, validation, and test sets following the rule of thumb 60/20/20 splits.
from sklearn.model_selection import train_test_split

X = df.drop(columns=['log_price'])
y = df['log_price']

# Step 1: Split off 20% for test set first
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 2: Split the remaining 80% into 60% train and 20% validation
# 0.25 x 80% = 20% of the full dataset
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print(f"Training set:    {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validation set:  {X_val.shape[0]:,} rows  ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:        {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X)*100:.0f}%)")

In [ ]:
# define the pipeline

# Linear Regression — baseline, no regularization
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

# Ridge — L2 regularization
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge())
])

# Lasso — L1 regularization
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Lasso())
])

In [38]:
# Establish baseline MSE for each model before any tuning
for name, pipeline in [('Linear Regression', lr_pipeline),
                        ('Ridge',             ridge_pipeline),
                        ('Lasso',             lasso_pipeline)]:

    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=5,
                             scoring='neg_mean_squared_error')

    mse_scores = -scores   # convert from negative to positive
    print(f"{name}:")
    print(f"  CV MSE scores: {mse_scores.round(4)}")
    print(f"  Mean MSE:      {mse_scores.mean():.4f}")
    print(f"  Std  MSE:      {mse_scores.std():.4f}")
    print()

KeyboardInterrupt: 

In [ ]:
# Tune Hyperparameeters
# --- Ridge ---
ridge_params = {'model__alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

ridge_finder = GridSearchCV(estimator=ridge_pipeline,
                            param_grid=ridge_params,
                            scoring='neg_mean_squared_error',
                            cv=5)
ridge_finder.fit(X_train, y_train)

print("Ridge Results:")
print(pd.DataFrame(ridge_finder.cv_results_)[['param_model__alpha',
                                               'mean_test_score',
                                               'std_test_score']])
print(f"Best alpha: {ridge_finder.best_params_}")
print(f"Best MSE:   {-ridge_finder.best_score_:.4f}")

In [ ]:
# Tune Hyperparameeters
# --- Lasso ---
lasso_params = {'model__alpha': [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]}

lasso_finder = GridSearchCV(estimator=lasso_pipeline,
                            param_grid=lasso_params,
                            scoring='neg_mean_squared_error',
                            cv=5)
lasso_finder.fit(X_train, y_train)

print("Lasso Results:")
print(pd.DataFrame(lasso_finder.cv_results_)[['param_model__alpha',
                                               'mean_test_score',
                                               'std_test_score']])
print(f"Best alpha: {lasso_finder.best_params_}")
print(f"Best MSE:   {-lasso_finder.best_score_:.4f}")

In [ ]:
# Fit Linear Regression first since it has no GridSearchCV
lr_pipeline.fit(X_train, y_train)

models = {
    'Linear Regression': lr_pipeline,
    'Ridge':             ridge_finder,
    'Lasso':             lasso_finder
}

print("=== Validation Set MSE ===\n")
for name, model in models.items():
    val_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, val_pred)
    print(f"{name}:  Validation MSE = {mse:.4f}")

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

#### Approved Model
Ridge Regression is selected as the final model based on the following:
- Matches Linear Regression and Lasso in predictive performance (MSE ~0.1820)
- Applies meaningful regularization (alpha=10) unlike Lasso (alpha=0.0001)
- Retains all features with shrunk coefficients for complete interpretation

In [ ]:
best_model = ridge_finder

test_pred = best_model.predict(X_test)
test_mse  = mean_squared_error(y_test, test_pred)
test_rmse = np.sqrt(test_mse)

print(f"=== Final Test Set Evaluation (Ridge) ===\n")
print(f"Test MSE  (log scale):    {test_mse:.4f}")
print(f"Test RMSE (log scale):    {test_rmse:.4f}")

# Convert back to dollar scale for business interpretation
test_pred_dollars = np.exp(test_pred)
y_test_dollars    = np.exp(y_test)
test_mse_dollars  = mean_squared_error(y_test_dollars, test_pred_dollars)
test_rmse_dollars = np.sqrt(test_mse_dollars)

print(f"\nTest MSE  (dollar scale): ${test_mse_dollars:,.0f}")
print(f"Test RMSE (dollar scale): ${test_rmse_dollars:,.0f}")

# Compare against validation MSE to confirm consistency
print(f"\n=== Consistency Check ===")
print(f"Validation MSE: 0.1820")
print(f"Test MSE:       {test_mse:.4f}")
gap = test_mse - 0.1820
print(f"Gap:            {gap:.4f}  ({'⚠️ investigate' if abs(gap) > 0.02 else '✅ consistent'})")

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.